# 분류 실습

**Classification**

입력이 속할 범주를 예측하는 지도학습 문제.

소재 분야에서 이해하기: 시료 이미지를 정상과 결함으로 구분한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 용어집](https://scikit-learn.org/stable/glossary.html)

## 1. 범주 예측

시료를 정상/결함으로 분류합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

n = 500
roughness = rng.uniform(0, 1, n)
porosity = rng.uniform(0, 1, n)
score = 3.2 * roughness + 2.6 * porosity - 2.4 + rng.normal(0, 0.45, n)
defect = (score > 0).astype(int)
features = np.column_stack([roughness, porosity])
print('결함 비율 %.1f%%' % (100 * defect.mean()))
plt.scatter(roughness, porosity, c=defect, cmap='coolwarm', s=14)
plt.xlabel('surface roughness (a.u.)'); plt.ylabel('porosity (a.u.)'); plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(features, defect, test_size=0.3, random_state=0, stratify=defect)
model = LogisticRegression().fit(X_train, y_train)
pred = model.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=['정상', '결함'], digits=3))

## 2. 판정 임계값을 바꿔보기

결함을 놓치지 않는 것이 중요하면 임계값을 낮춥니다.

In [ ]:
probability = model.predict_proba(X_test)[:, 1]
for threshold in (0.3, 0.5, 0.7):
    flagged = (probability > threshold).astype(int)
    tp = int(((flagged == 1) & (y_test == 1)).sum()); fn = int(((flagged == 0) & (y_test == 1)).sum())
    fp = int(((flagged == 1) & (y_test == 0)).sum())
    print('임계값 %.1f -> 놓친 결함 %2d개, 잘못 잡은 정상 %2d개, 잡아낸 결함 %2d개' % (threshold, fn, fp, tp))

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#classification)을 여세요.